In [2]:
# ==============================
# Cell 1: Imports and Setup
# ==============================

import os
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from joblib import load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ==============================
# Cell 2: Paths & Config
# ==============================

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64
BATCH_SIZE = 16

TEST_PATH = "data/test.csv"
BEST_MODEL_PATH = "models/best_model.pt"
LABEL_ENCODER_PATH = "models/label_encoder.joblib"

print("Current directory:", os.getcwd())
print("Files:", os.listdir())
if os.path.exists("data"):
    print("data/ contents:", os.listdir("data"))


# ==============================
# Cell 3: Load Data and Label Encoder
# ==============================

test_df = pd.read_csv(TEST_PATH)
print("Test data sample:")
print(test_df.head())

label_encoder = load(LABEL_ENCODER_PATH)
print("Label classes:", label_encoder.classes_)

X_test = test_df["text"].values
y_test = test_df["label_id"].values


# ==============================
# Cell 4: Dataset Class and Tokenizer
# ==============================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

test_dataset = IntentDataset(X_test, y_test, tokenizer, MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# ==============================
# Cell 5: Load Model
# ==============================

num_labels = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.to(device)
model.eval()


# ==============================
# Cell 6: Evaluate + Confusion Matrix
# ==============================

all_preds = []
all_labels = []
total_loss = 0.0

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        total_loss += loss.item()

        _, preds = torch.max(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_loss = total_loss / len(test_loader)
acc = accuracy_score(all_labels, all_preds)

print(f"Test loss: {avg_loss:.4f}")
print(f"Test accuracy: {acc:.4f}")

report = classification_report(
    all_labels,
    all_preds,
    target_names=label_encoder.classes_
)
print("Test classification report:")
print(report)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

Using device: cpu
Current directory: d:\videos\Project
Files: ['data', 'models', 'results', 'test.ipynb', 'Training.ipynb']
data/ contents: ['intent_questions.csv', 'test.csv', 'train.csv', 'val.csv', 'validation.ipynb']
Test data sample:
                                         text  label_id
0  How often should I irrigate my maize crop?         3
1       Recommended fertilizer dose for rice?         1
2    How do I control pests in my rice field?         8
3     When should I water tomatoes in summer?         4
4  How can I prevent fungal disease in wheat?         0
Label classes: ['disease_control_wheat' 'fertilizer_rice' 'fertilizer_wheat'
 'irrigation_maize' 'irrigation_vegetables' 'irrigation_wheat'
 'pest_control_cotton' 'pest_control_maize' 'pest_control_rice']


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 177.41it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FileNotFoundError: [Errno 2] No such file or directory: 'models/best_model.pt'